# Pipeline Principal de Análisis

Este notebook ejecuta el pipeline completo de procesamiento y análisis de datos del experimento de personalidad implícita y cambio de opinión política.

## Estructura del Pipeline

### Preprocesamiento (11 pasos):
1. Construcción de bases de datos
2. Cálculo de índices ideológicos
3. Creación de variables de cambio (CO y CT)
4. Limpieza de datos y eliminación de outliers
5. Relleno de medianas por categoría ideológica
6. Procesamiento de redes sociales y medios
7. Agrupamiento de variables sociodemográficas
8. Creación de variables dummy
9. Ordenamiento de columnas
10. Agregado de clusters
11. Exportación de bases finales

### Análisis Estadístico:
- Tests de Mann-Whitney
- Análisis de congruencia ideológica
- Modelos SEM
- Correlaciones de Spearman

### Visualizaciones:
- Gráficos de Cleveland
- Heatmaps de correlación
- Gráficos de violín

---

## Opción 1: Ejecutar Pipeline Completo Automatizado

**Recomendado para ejecución rápida del pipeline completo.**

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================

import sys
from pathlib import Path

# Agregar rutas de módulos al path.
Ruta_Base = Path.cwd().parent

# Importar script del pipeline.
sys.path.append(str(Ruta_Base))

from Ejecutar_Pipeline_Completo import (
    Ejecutar_Pipeline_Completo
)

# Ejecutar pipeline completo.
Diccionario_Dfs = Ejecutar_Pipeline_Completo(
    Guardar_Intermedios=False,
    Verbose=True
)

# Extraer DataFrames.
Df_Generales = Diccionario_Dfs['Generales']
Df_Ballotage = Diccionario_Dfs['Ballotage']

print("\n✅ Pipeline completado. DataFrames listos para análisis.")

---

## Opción 2: Ejecutar Pipeline Paso a Paso

**Recomendado para exploración y debugging.**

### Configuración Inicial

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================

import sys
from pathlib import Path

# Agregar rutas de módulos al path.
Ruta_Base = Path.cwd().parent
sys.path.append(str(Ruta_Base / "Codigo" / "Utilidades"))
sys.path.append(str(Ruta_Base / "Codigo" / "Procesamiento"))
sys.path.append(str(Ruta_Base / "Codigo" / "Test_Estadisticos"))
sys.path.append(
    str(Ruta_Base / "Codigo" / "Modelado_Estadistico")
)
sys.path.append(str(Ruta_Base / "Codigo" / "Visualizacion"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from Configuracion import *
from Funciones_Comunes import *

print("✓ Módulos importados correctamente.")

### Paso 1: Construcción de Bases de Datos

In [ ]:
from Construccion_Bases import (
    Combinar_Archivos_Generales,
    Combinar_Archivos_Ballotage,
    Procesar_Base_Completa
)

# Cargar y procesar datos de Generales.
print("\nCargando datos de Generales...")
Df_Generales_Crudo = Combinar_Archivos_Generales()
Df_Generales = Procesar_Base_Completa(Df_Generales_Crudo)

# Cargar y procesar datos de Ballotage.
print("Cargando datos de Ballotage...")
Df_Ballotage_Crudo = Combinar_Archivos_Ballotage()
Df_Ballotage = Procesar_Base_Completa(Df_Ballotage_Crudo)

print(f"\n✓ Datos Generales: {len(Df_Generales)} participantes")
print(f"✓ Datos Ballotage: {len(Df_Ballotage)} participantes")

# Crear diccionario.
Diccionario_Dfs = {
    'Generales': Df_Generales,
    'Ballotage': Df_Ballotage
}

### Paso 2: Cálculo de Índices Ideológicos

In [ ]:
from Calculo_Indices import Calcular_Todos_Indices

# Calcular índices para ambas bases.
print("\nCalculando índices ideológicos...")
for Nombre, Df in Diccionario_Dfs.items():
    Diccionario_Dfs[Nombre] = Calcular_Todos_Indices(Df)

print("\n✓ Índices calculados.")

# Mostrar estadísticas descriptivas.
print("\nEstadísticas de Índices (Generales):")
print(Diccionario_Dfs['Generales'][[
    'Indice_Progresismo',
    'Indice_Conservadurismo',
    'Indice_Positividad'
]].describe())

### Paso 3: Creación de Variables de Cambio (CO y CT)

In [ ]:
from Calculo_Variables_Cambio import Calcular_Variables_Cambio

# Calcular variables CO y CT.
print("\nCreando variables de cambio...")
Diccionario_Dfs = Calcular_Variables_Cambio(
    Diccionario_Dfs,
    Incluir_Congruencia=True
)

print("\n✓ Variables de cambio creadas.")
print(f"  Variables CO: 40 por base")
print(f"  Variables CT: 40 por base")

### Paso 4: Limpieza de Datos y Eliminación de Outliers

In [ ]:
from Limpieza_Datos import Limpiar_Diccionario_Dataframes

# Guardar tamaños pre-limpieza.
Tamaños_Pre = {
    Nombre: len(Df) for Nombre, Df in Diccionario_Dfs.items()
}

# Limpiar ambos DataFrames.
print("\nLimpiando datos...")
Diccionario_Dfs = Limpiar_Diccionario_Dataframes(
    Diccionario_Dfs,
    Filtrar_Categorias=True,
    Filtrar_Tiempos=True,
    Numero_Desviaciones=3
)

# Mostrar resumen.
print("\n✓ Datos limpios:")
for Nombre, Df in Diccionario_Dfs.items():
    Eliminados = Tamaños_Pre[Nombre] - len(Df)
    Porcentaje = (Eliminados / Tamaños_Pre[Nombre]) * 100
    print(
        f"  {Nombre}: {len(Df)} casos "
        f"({Eliminados} eliminados, {Porcentaje:.1f}%)"
    )

### Paso 5: Relleno de Medianas

In [ ]:
from Relleno_Medianas import (
    Rellenar_Con_Medianas_Por_Categoria
)

print("\nRellenando valores faltantes con medianas...")
for Nombre, Df in Diccionario_Dfs.items():
    Diccionario_Dfs[Nombre] = (
        Rellenar_Con_Medianas_Por_Categoria(
            Df,
            Columna_Categoria='Categoria_PASO_2023'
        )
    )

print("\n✓ Medianas aplicadas.")

### Paso 6: Procesamiento de Redes y Medios

In [ ]:
from Procesar_Redes_Y_Medios import (
    Procesar_Redes_Sociales,
    Procesar_Medios_Prensa
)

print("\nProcesando redes sociales y medios...")
for Nombre, Df in Diccionario_Dfs.items():
    Df = Procesar_Redes_Sociales(Df)
    Df = Procesar_Medios_Prensa(Df)
    Diccionario_Dfs[Nombre] = Df

print("\n✓ Redes y medios procesados.")

### Paso 7: Agrupamiento de Variables

In [ ]:
from Agrupamiento_Variables import (
    Agrupar_Edad,
    Mapear_Provincia_A_Region,
    Agrupar_Nivel_Educativo
)

print("\nAgrupando variables sociodemográficas...")
for Nombre, Df in Diccionario_Dfs.items():
    if 'Edad' in Df.columns:
        Df['Edad_Agrupada'] = Agrupar_Edad(Df['Edad'])

    if 'Provincia' in Df.columns:
        Df['Region'] = Mapear_Provincia_A_Region(Df['Provincia'])

    if 'Nivel_Educativo' in Df.columns:
        Df['Nivel_Educativo_Agrupado'] = (
            Agrupar_Nivel_Educativo(Df['Nivel_Educativo'])
        )

    Diccionario_Dfs[Nombre] = Df

print("\n✓ Agrupamientos aplicados.")

### Paso 8: Creación de Variables Dummy

In [ ]:
from Crear_Variables_Dummy import Crear_Todas_Variables_Dummy

print("\nCreando variables dummy...")
for Nombre, Df in Diccionario_Dfs.items():
    Diccionario_Dfs[Nombre] = Crear_Todas_Variables_Dummy(Df)

print("\n✓ Variables dummy creadas.")

### Paso 9: Ordenamiento de Columnas

In [ ]:
from Ordenamiento_Columnas import Ordenar_Columnas_Por_Tematica

print("\nOrdenando columnas...")
for Nombre, Df in Diccionario_Dfs.items():
    Diccionario_Dfs[Nombre] = (
        Ordenar_Columnas_Por_Tematica(Df)
    )

print("\n✓ Columnas ordenadas.")

### Paso 10: Agregado de Clusters

In [ ]:
from Agregar_Clusters import Agregar_Clusters_A_Base

print("\nAgregando resultados de clustering...")
Ruta_Clusters = RUTA_DATOS_CRUDOS / "Resultados_Clustering"

if Ruta_Clusters.exists():
    for Nombre, Df in Diccionario_Dfs.items():
        try:
            Diccionario_Dfs[Nombre] = (
                Agregar_Clusters_A_Base(Df, str(Ruta_Clusters))
            )
        except FileNotFoundError:
            print(
                f"⚠️ {Nombre}: Archivos de clustering no encontrados"
            )

    print("\n✓ Clusters agregados.")
else:
    print("\n⚠️ Carpeta de clustering no encontrada. Saltando paso.")

### Paso 11: Exportación de Bases Finales

In [ ]:
print("\nExportando bases finales...")

Ruta_Export = Path(RUTA_BASES_DEFINITIVAS)
Ruta_Export.mkdir(parents=True, exist_ok=True)

for Nombre, Df in Diccionario_Dfs.items():
    Archivo_Salida = (
        Ruta_Export / f"Bases finales{Nombre}.xlsx"
    )

    Df.to_excel(Archivo_Salida, index=False)
    print(f"  ✓ {Nombre}: {Archivo_Salida.name}")

print("\n✅ Pipeline de preprocesamiento completado.")

---

## Análisis Estadístico

### Test de Mann-Whitney (Ejemplo)

In [ ]:
from Mann_Whitney import Ejecutar_Mann_Whitney

# Ejemplo: Comparar CO_Item_3_Izq entre grupos ideológicos.
Resultado = Ejecutar_Mann_Whitney(
    Diccionario_Dfs['Generales'],
    'CO_Item_3_Izq',
    ['Left_Wing', 'Progressivism'],
    ['Right_Wing_Libertarian']
)

print("\nResultado del test Mann-Whitney:")
print(f"  P-valor: {Resultado['P_Valor']:.4f}")
print(f"  Significativo: {Resultado['Significativo']}")
print(f"  Tamaño del efecto: {Resultado['Tamaño_Efecto']:.3f}")

### Análisis de Congruencia Ideológica

In [ ]:
from Congruencia_Ideologica import (
    Analizar_Congruencia_Por_Candidato
)

# Analizar congruencia para candidatos principales.
Resultados_Congruencia = Analizar_Congruencia_Por_Candidato(
    Diccionario_Dfs['Generales'],
    Lista_Candidatos=['Milei', 'Bullrich', 'Bregman', 'Solano']
)

print("\nResultados de congruencia ideológica:")
print(Resultados_Congruencia)

### Modelos SEM (Ejemplo)

In [ ]:
from Modelos_SEM import Ejecutar_Modelo_SEM_Simple

# Ejemplo: Predecir CO con Índice de Progresismo.
Resultado_SEM = Ejecutar_Modelo_SEM_Simple(
    Diccionario_Dfs['Generales'],
    'Indice_Progresismo',
    'CO_Item_3_Izq'
)

if Resultado_SEM['Exito']:
    print("\nResultado del modelo SEM:")
    print(f"  Coeficiente: {Resultado_SEM['Coeficiente']:.3f}")
    print(f"  P-valor: {Resultado_SEM['P_Valor']:.4f}")
    print(f"  R²: {Resultado_SEM['R2']:.3f}")

---

## Visualizaciones

### Distribución de Índices por Categoría Ideológica

In [ ]:
# Boxplot de Índice de Progresismo por categoría.
Fig, Ax = plt.subplots(figsize=(12, 6))

Diccionario_Dfs['Generales'].boxplot(
    column='Indice_Progresismo',
    by='Categoria_PASO_2023',
    ax=Ax
)

Ax.set_title(
    'Índice de Progresismo por Categoría Ideológica (Generales)',
    fontsize=14
)
Ax.set_xlabel('Categoría PASO 2023', fontsize=12)
Ax.set_ylabel('Índice de Progresismo', fontsize=12)
Ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Correlación entre Índices

In [ ]:
# Matriz de correlación entre índices.
Indices = [
    'Indice_Progresismo',
    'Indice_Conservadurismo',
    'Indice_Positividad'
]

Matriz_Corr = (
    Diccionario_Dfs['Generales'][Indices].corr(method='spearman')
)

Fig, Ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    Matriz_Corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=Ax
)

Ax.set_title(
    'Correlación entre Índices (Spearman)',
    fontsize=14
)

plt.tight_layout()
plt.show()

---

## Resumen Final

In [ ]:
print("\n" + "="*70)
print("RESUMEN FINAL DEL PIPELINE")
print("="*70)

for Nombre, Df in Diccionario_Dfs.items():
    print(f"\n{Nombre}:")
    print(f"  Participantes: {len(Df)}")
    print(f"  Columnas: {len(Df.columns)}")
    print(
        f"  Variables CO: "
        f"{len([c for c in Df.columns if c.startswith('CO_')])}"
    )
    print(
        f"  Variables CT: "
        f"{len([c for c in Df.columns if c.startswith('CT_')])}"
    )
    print(
        f"  Memoria: "
        f"{Df.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
    )

print("\n" + "="*70)
print("✅ ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*70)